In [ ]:
import requests
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
print(len(text), "characters")
print(text[:200])

1115394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [ ]:
# unique characters dhundho
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size)
print(''.join(chars))

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [ ]:
# char to int aur int to char mappings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# test karo
print(encode("hello"))
print(decode(encode("hello")))

[46, 43, 50, 50, 53]
hello


In [ ]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape)
print(data[:20])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56])


In [ ]:
print(decode(data[:20].tolist()))


First Citizen:
Befor


In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(len(train_data))
print(len(val_data))

1003854
111540


In [ ]:
block_size = 8  # ek baar mein kitne characters context mein dekhega
x = train_data[:block_size]
y = train_data[1:block_size+1]

for i in range(block_size):
    context = x[:i+1]
    target = y[i]
    print(f"context: {context.tolist()} --> target: {target}")

context: [18] --> target: 47
context: [18, 47] --> target: 56
context: [18, 47, 56] --> target: 57
context: [18, 47, 56, 57] --> target: 58
context: [18, 47, 56, 57, 58] --> target: 1
context: [18, 47, 56, 57, 58, 1] --> target: 15
context: [18, 47, 56, 57, 58, 1, 15] --> target: 47
context: [18, 47, 56, 57, 58, 1, 15, 47] --> target: 58


In [ ]:
torch.manual_seed(1337)
batch_size = 4   # ek saath kitne sequences process karein
block_size = 8   # har sequence ki length

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print("inputs shape:", xb.shape)
print("targets shape:", yb.shape)
print(xb)

inputs shape: torch.Size([4, 8])
targets shape: torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(5.0364, grad_fn=<NllLossBackward0>)


In [ ]:
idx = torch.zeros((1,1), dtype=torch.long)  # starting token = 0
generated = m.generate(idx, max_new_tokens=100)[0].tolist()
print(decode(generated))


lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheH


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if steps % 1000 == 0:
        print(f"step {steps}: loss {loss.item():.4f}")

print("Final loss:", loss.item())

step 0: loss 4.6477
step 1000: loss 3.6650
step 2000: loss 3.3166
step 3000: loss 2.8780
step 4000: loss 2.6928
step 5000: loss 2.4840
step 6000: loss 2.5045
step 7000: loss 2.5102
step 8000: loss 2.4093
step 9000: loss 2.4303
Final loss: 2.362441062927246


In [ ]:
idx = torch.zeros((1,1), dtype=torch.long)
generated = m.generate(idx, max_new_tokens=300)[0].tolist()
print(decode(generated))


Whod
Wht s

MI wect!-lltherotheve t fe;
WAnd py;

PO t s ld tathat, ir V
IO thesecin teot tit ado ilorer.
Ply, d'stacoes, ld omat mealellly yererer EMEvesas ie IZEd pave mautoofareanerllleyomerer but?
The t,
Ith'dwitile w? beren to'd ff a atrts brey s

ESesenther:
Ithon f at par,
NTmamy an flictong 


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Pichle session ke hyperparameters
batch_size = 32
block_size = 8
n_embd = 32  # embedding dimension

class Head(nn.Module):
    """ Single self-attention head """

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Mask — future tokens nahi dekhne ke liye
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape  # Batch, Time(tokens), Channels(embedding)

        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # Attention scores
        wei = q @ k.transpose(-2, -1) * C**-0.5  # (B, T, T)

        # Mask — future block karo
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # Softmax
        wei = F.softmax(wei, dim=-1)  # (B, T, T)

        # Weighted aggregation
        v = self.value(x)  # (B, T, head_size)
        out = wei @ v      # (B, T, head_size)
        return out

In [ ]:
# Test karte hain
x = torch.randn(batch_size, block_size, n_embd)  # fake input
print("Input shape:", x.shape)  # (32, 8, 32)

head = Head(head_size=16)
out = head(x)
print("Output shape:", out.shape)  # (32, 8, 16)

Input shape: torch.Size([32, 8, 32])
Output shape: torch.Size([32, 8, 16])


In [ ]:
class MultiHeadAttention(nn.Module):
    """ Multiple heads running in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)  # projection layer

    def forward(self, x):
        # Har head alag alag chalao, phir concat karo
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [ ]:
# 4 heads, har head 8-dim (4*8 = 32 = n_embd)
mha = MultiHeadAttention(num_heads=4, head_size=8)
out = mha(x)
print("MultiHead output shape:", out.shape)  # (32, 8, 32)

MultiHead output shape: torch.Size([32, 8, 32])


In [ ]:
class FeedForward(nn.Module):
    """ Simple MLP — har token independently process hota hai """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # expand
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),  # contract
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
ff = FeedForward(n_embd)
out = ff(x)
print("FeedForward output shape:", out.shape)  # (32, 8, 32)

FeedForward output shape: torch.Size([32, 8, 32])


In [ ]:
class Block(nn.Module):
    """ Full Transformer Block """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))  # attention + residual
        x = x + self.ff(self.ln2(x))  # feedforward + residual
        return x

In [ ]:
block = Block(n_embd=32, n_head=4)
out = block(x)
print("Block output shape:", out.shape)  # (32, 8, 32)

Block output shape: torch.Size([32, 8, 32])


In [ ]:
class GPTLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            nn.LayerNorm(n_embd),
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Embeddings
        tok_emb = self.token_embedding_table(idx)  # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T))  # (T, n_embd)
        x = tok_emb + pos_emb  # (B, T, n_embd)

        # Transformer blocks
        x = self.blocks(x)

        # Output
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # Block size tak crop karo
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]  # last token
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
# vocab_size pichle session se — 65 tha Shakespeare ka
model = GPTLanguageModel(vocab_size=65)
logits, loss = model(torch.zeros((1, 8), dtype=torch.long))
print("Logits shape:", logits.shape)  # (8, 65) — untrained

Logits shape: torch.Size([1, 8, 65])


In [ ]:
# Naya GPT model — fresh
model = GPTLanguageModel(vocab_size=65)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for steps in range(5000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if steps % 500 == 0:
        print(f"Step {steps} | Loss: {loss.item():.4f}")

print("Final loss:", loss.item())

Step 0 | Loss: 4.3559
Step 500 | Loss: 2.3877
Step 1000 | Loss: 2.2028
Step 1500 | Loss: 2.2682
Step 2000 | Loss: 2.1112
Step 2500 | Loss: 2.0929
Step 3000 | Loss: 1.9991
Step 3500 | Loss: 2.1323
Step 4000 | Loss: 2.0220
Step 4500 | Loss: 1.9258
Final loss: 1.9555964469909668


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=300)
print(''.join(itos[i] for i in generated[0].tolist()))


Which's for disents as alumnot, or and wee earen:
For be colvefite;
Whough lould your there on the heer sonben, hing,
Of-feew, afe and ashees, not hany his whichs have sown pown chentrongees offer'sgowld up gratent God is proyoe glard I condsore upchason
Of argaing?

KING INCEordengs, giverer, died

